# Стратегии чанкинга: эксперименты и сравнение поиска по разным БД

Песочница по статье [7 Chunking Strategies That Decide Whether Your RAG Works](https://machinelearningmastery.com/7-chunking-strategies-that-decide-whether-your-rag-works/).
Корпус: **markdown-инструкции поддержки** (в одном файле может быть несколько инструкций).
Цель: нарезать документы разными стратегиями на **langchain**, собрать **отдельную экспериментальную БД под каждую стратегию** и сравнить поиск по одним и тем же вопросам.

| Стратегия | Статья | Реализация |
|---|---|---|
| `baseline-1200-120` | контроль | продакшен `split_text` проекта |
| `fixed-512-64` | #1 fixed-size + overlap | `CharacterTextSplitter(separator="")` |
| `sentence-window` | #2 small-to-big | мелкие окна предложений; расширение контекста уже делает гидратор |
| `markdown-recursive` | #3 structural | `RecursiveCharacterTextSplitter.from_language(MARKDOWN)` |
| `markdown-headers` | #3 structural | `MarkdownHeaderTextSplitter` + хлебные крошки заголовков |
| `markdown-blocks` | #7 table-preserving | атомарные блоки: таблицы и code-fence не режутся |
| `semantic-ollama` | #4 semantic | `SemanticChunker` (langchain-experimental) на эмбеддингах Ollama |

Не включены: #5 parent-child и #6 agentic LLM — первая требует merge на стороне retrieval
(проектный `SearchService` этого не делает), вторая недетерминированна и дорога.

**Границы записи:** продакшен-дамп и продакшен-БД только читаются. Все артефакты
(вариантные дампы, .lbdb, run.json/results.json) пишутся в `runtime/debug/chunking-experiments/`.
Код проекта ноутбук не меняет.

Порядок: Run All сверху вниз. Настройки — только в ячейке CONFIG.

## 1. Корень проекта и ядро

Ноутбук должен работать на ядре основного окружения проекта `mvp_v2/.venv`
(python из `mvp_v2/.venv`).

In [13]:
from __future__ import annotations

import importlib
import json
import os
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/mvp_v2").is_dir():
            return candidate
        nested = candidate / "mvp_v2"
        if (nested / "pyproject.toml").is_file() and (nested / "src/mvp_v2").is_dir():
            return nested
    raise RuntimeError("Run this notebook from the repository or mvp_v2 directory")


PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))


load_dotenv(PROJECT_ROOT / ".env", override=True)
print("Project:", PROJECT_ROOT)
print("Kernel:", sys.executable)

Project: /Users/zero/GITHUB/vue-graph-rag/mvp_v2
Kernel: /Users/zero/GITHUB/vue-graph-rag/mvp_v2/.venv/bin/python


In [14]:
sys.path.insert(0, str(PROJECT_ROOT / "src/mvp_v2"))

## 2. CONFIG - единственная ячейка настроек

Меняйте только эту ячейку. По умолчанию собираются все 7 стратегий по всем
текущим `.md`-документам; для быстрого прогона уберите стратегии из `STRATEGIES`,
поставьте `MAX_DOCUMENTS = 5` или выключите `BUILD_DATABASES` (только статистика чанков).

In [15]:

# ===================== CONFIG: меняйте только здесь =====================

# Исходная база: None = самая свежая проверенная .lbdb из runtime/databases,
# иначе имя базы без суффикса, например "rag-db".
SOURCE_DATABASE_NAME: str | None = None

# Стратегии чанкинга: одна экспериментальная БД на стратегию.
# "baseline-1200-120" - контроль: та же нарезка, что в продакшен-инжесте.
STRATEGIES: tuple[str, ...] = (
    "baseline-1200-120",
    "fixed-512-64",
    "sentence-window",
    "markdown-recursive",
    "markdown-headers",
    "markdown-blocks",
    "semantic-ollama",
)

# Параметры стратегий (размеры в символах, как и в продакшен-политике проекта).
STRATEGY_PARAMS: dict[str, dict[str, Any]] = {
    "baseline-1200-120": {"max_characters": 1_200, "overlap_characters": 120},
    "fixed-512-64": {"chunk_size": 512, "chunk_overlap": 64},
    "sentence-window": {"max_sentences": 3, "overlap_sentences": 1},
    "markdown-recursive": {"chunk_size": 1_200, "chunk_overlap": 120},
    "markdown-headers": {"chunk_size": 1_200, "chunk_overlap": 120},
    "markdown-blocks": {"max_characters": 1_200},
    "semantic-ollama": {"breakpoint_threshold_amount": 95.0},
}

# Документы: None = все текущие .md-версии; число = первые N (быстрый прогон).
MAX_DOCUMENTS: int | None = None

# Вопросы для сравнения поиска (одни и те же вопросы -> все стратегии).
MY_QUESTIONS: tuple[str, ...] = (
    "Что нужно для того, чтобы мы начали работу над твоим резюме?",
    "Где купить прокси?",
    "Сколько рабочих почт, номеров, SIM-карт, HH и Telegram нужно создать?",
)

EVIDENCE_LIMIT = 3               # топ-N доказательств (1..100)
MINIMUM_SIMILARITY_SCORE = 0.55  # порог косинус-близости (-1..1)

# Этапы прогона: False = только превью статистики чанков (быстро, без Ollama).
BUILD_DATABASES = True           # собрать вариантные дампы и .lbdb (зовёт Ollama-эмбеддинги)
RUN_SEARCH = True                # прогнать MY_QUESTIONS по собранным БД

PREVIEW_CHARACTERS = 200         # сколько символов чанка показывать в превью
GENERATION_TEMPERATURE = 0.0
OLLAMA_TIMEOUT_SECONDS = 120.0

# Куда писать: runtime/debug/chunking-experiments/<дата>/<run_id>/{dumps,db,...}.
RUNS_ROOT = PROJECT_ROOT / "runtime/debug/chunking-experiments"

# --- проверки значений ---
unknown = [name for name in STRATEGIES if name not in STRATEGY_PARAMS]
if unknown:
    raise ValueError(f"Неизвестные стратегии в STRATEGIES: {unknown}")
if not STRATEGIES:
    raise ValueError("STRATEGIES пуст: выберите хотя бы одну стратегию")
if MAX_DOCUMENTS is not None and MAX_DOCUMENTS < 1:
    raise ValueError("MAX_DOCUMENTS: None или >= 1")
if not MY_QUESTIONS or any(not question.strip() for question in MY_QUESTIONS):
    raise ValueError("MY_QUESTIONS: нужен хотя бы один непустой вопрос")
if not 1 <= EVIDENCE_LIMIT <= 100:
    raise ValueError("EVIDENCE_LIMIT должен быть от 1 до 100")
if not -1.0 <= MINIMUM_SIMILARITY_SCORE <= 1.0:
    raise ValueError("MINIMUM_SIMILARITY_SCORE должен быть в диапазоне [-1, 1]")
print(
    "CONFIG ok:",
    len(STRATEGIES), "стратегий;",
    "документов", "все" if MAX_DOCUMENTS is None else MAX_DOCUMENTS,
    "; сборка БД", "вкл" if BUILD_DATABASES else "выкл",
    "; поиск", "вкл" if RUN_SEARCH else "выкл",
)

CONFIG ok: 7 стратегий; документов все ; сборка БД вкл ; поиск вкл


## 3. Импорты и reload (контракты до сервисов)

Контракты (DTO) перезагружаются ДО сервисов: иначе в живом ядре остаются
устаревшие Pydantic-классы. После правок кода mvp_v2 просто снова Run All.
LangChain-сплиттеры - из зависимостей проекта (`langchain-text-splitters`,
`langchain-experimental`).

In [16]:
# Порядок важен: контракты и домен -> подготовка -> адаптеры -> сервисы -> компоновка.
import mvp_v2.application.ingestion_contracts as ingestion_contracts_module
import mvp_v2.application.dto.retrieval as retrieval_dto_module
import mvp_v2.application.dump_lifecycle.manifests as manifests_module
import mvp_v2.domain.graph as graph_module
import mvp_v2.domain.snapshot as snapshot_module
import mvp_v2.application.rag.preparation.chunking as chunking_module
import mvp_v2.application.rag.preparation.chunk_backfill as chunk_backfill_module
import mvp_v2.adapters.filesystem.media as media_module
import mvp_v2.adapters.filesystem.dumps as dumps_module
import mvp_v2.adapters.ladybug as ladybug_adapter_module
import mvp_v2.adapters.ollama as ollama_module
import mvp_v2.application.rag.preparation.model_operations as model_operations_module
import mvp_v2.application.dump_lifecycle.publication as publication_module
import mvp_v2.application.dump_lifecycle.validation as validation_module
import mvp_v2.application.rag.retrieval.retrieval as retrieval_module
import mvp_v2.bootstrap.ui as bootstrap_ui_module
import mvp_v2.streamlit_ui.controller as controller_module

_RELOAD_ORDER = (
    ingestion_contracts_module,  # контракты
    retrieval_dto_module,
    manifests_module,
    graph_module,                # домен
    snapshot_module,
    chunking_module,             # подготовка
    chunk_backfill_module,
    media_module,                # адаптеры
    dumps_module,
    ladybug_adapter_module,
    ollama_module,
    model_operations_module,     # сервисы
    publication_module,
    validation_module,
    retrieval_module,
    bootstrap_ui_module,         # компоновка (зависит от адаптеров)
    controller_module,
)
for module in _RELOAD_ORDER:
    importlib.reload(module)

# Свежие имена из перезагруженных модулей.
ModelAuthorization = ingestion_contracts_module.ModelAuthorization
ModelIdentity = ingestion_contracts_module.ModelIdentity
EmbeddingRequest = ingestion_contracts_module.EmbeddingRequest
SearchRequest = retrieval_dto_module.SearchRequest
EmbeddingPurpose = retrieval_dto_module.EmbeddingPurpose
graph = graph_module
GraphSnapshot = snapshot_module.GraphSnapshot
ChunkPolicy = chunking_module.ChunkPolicy
split_text = chunking_module.split_text
make_chunks = chunking_module.make_chunks
sanitize_source_text = chunk_backfill_module._sanitize_source_text
FilesystemMediaResolver = media_module.FilesystemMediaResolver
FilesystemDumpRepository = dumps_module.FilesystemDumpRepository
load_verified_dump = ladybug_adapter_module.load_verified_dump
build_database = ladybug_adapter_module.build_database
LadybugAnn = ladybug_adapter_module.LadybugAnn
LocalHttpTransport = ollama_module.LocalHttpTransport
OllamaAdapter = ollama_module.OllamaAdapter
build_embeddings = model_operations_module.build_embeddings
SnapshotPublicationService = publication_module.SnapshotPublicationService
validate_snapshot = validation_module.validate_snapshot
SearchService = retrieval_module.SearchService
default_runtime_config = bootstrap_ui_module.default_runtime_config
configure_ladybug_home = controller_module.configure_ladybug_home
scan_database_manifests = controller_module.scan_database_manifests
database_is_stale = controller_module.database_is_stale

# LangChain-сплиттеры (зависимости проекта: langchain-text-splitters,
# langchain-experimental). SemanticChunker тянет langchain_core.
try:
    from langchain_text_splitters import (
        CharacterTextSplitter,
        Language,
        MarkdownHeaderTextSplitter,
        RecursiveCharacterTextSplitter,
    )
    from langchain_core.embeddings import Embeddings
    from langchain_experimental.text_splitter import SemanticChunker
except ImportError as error:
    raise RuntimeError(
        "Нужны langchain-text-splitters и langchain-experimental: "
        "выполните uv sync для mvp_v2"
    ) from error

# Защита от устаревших типов в живом ядре.
if "minimum_score" not in SearchRequest.model_fields:
    raise RuntimeError("SearchRequest устарел: перезапустите ядро и Run All")
if "generation_token" not in EmbeddingRequest.model_fields:
    raise RuntimeError("EmbeddingRequest устарел: перезапустите ядро и Run All")
print("Модули загружены и перезагружены в порядке зависимостей")

Модули загружены и перезагружены в порядке зависимостей


## 4. Исходная БД, дамп и документы (read-only)

Берём проверенную БД и её исходный дамп, читаем тексты текущих `.md`-версий
и готовим Ollama-адаптер (он нужен и для семантической нарезки, и для
эмбеддингов новых чанков). Продакшен-данные не меняются.

In [17]:
config = default_runtime_config()
verified_databases = scan_database_manifests(config.databases_root)
if not verified_databases:
    raise RuntimeError(
        "Проверенные БД не найдены в "
        + str(config.databases_root)
        + "; постройте базу через Streamlit или Telegram"
    )

if SOURCE_DATABASE_NAME is None:
    database_path, database_manifest = max(
        verified_databases, key=lambda item: item[1].built_at
    )
    print("SOURCE_DATABASE_NAME = None -> выбрана новейшая БД по built_at")
else:
    wanted = (
        SOURCE_DATABASE_NAME
        if SOURCE_DATABASE_NAME.endswith(".lbdb")
        else SOURCE_DATABASE_NAME + ".lbdb"
    )
    selected = [item for item in verified_databases if item[0].name == wanted]
    if len(selected) != 1:
        raise RuntimeError(
            f"Проверенная БД {wanted!r} не найдена однозначно в {config.databases_root}"
        )
    database_path, database_manifest = selected[0]

source_dump_path = (
    config.dumps_root / f"dump-{database_manifest.source_dump_number:06d}"
)
dump_manifest, snapshot = load_verified_dump(source_dump_path)
if database_is_stale(database_manifest, dump_manifest):
    raise RuntimeError("Манифест БД не совпадает с исходным дампом")

print("Database   :", database_path.name)
print("Source dump:", source_dump_path.name, "run", dump_manifest.run_id)
print("Space      :", database_manifest.embedding_space.model_dump(mode="json"))
print(
    "Snapshot   : документов", len(snapshot.documents),
    "| версий", len(snapshot.document_versions),
    "| чанков", len(snapshot.chunks),
    "| эмбеддингов", len(snapshot.embeddings),
)

# --- текущие .md-версии и их тексты ---
resolver = FilesystemMediaResolver(config.working_root)
current_md_versions = sorted(
    (
        version
        for version in snapshot.document_versions
        if version.is_current
        and version.local_path
        and version.local_path.endswith(".md")
    ),
    key=lambda version: version.local_path or "",
)
if not current_md_versions:
    raise RuntimeError("В дампе нет текущих .md-версий документов")
if MAX_DOCUMENTS is not None:
    current_md_versions = current_md_versions[:MAX_DOCUMENTS]
SELECTED_VERSIONS = tuple(current_md_versions)

document_titles = {document.id: document.title for document in snapshot.documents}
version_title = {
    edge.target_id: document_titles.get(edge.source_id, edge.source_id)
    for edge in snapshot.has_versions
}
SOURCES: dict[str, str] = {}
for version in SELECTED_VERSIONS:
    raw_text = resolver.read_verified(version.local_path, version.sha256).decode("utf-8")
    # Контракт Chunk разрешает только LF/TAB: санитизируем, как chunk_backfill в проде.
    SOURCES[version.id] = sanitize_source_text(raw_text)
    if SOURCES[version.id] != raw_text:
        print(f"санитизирован (control-символы): {version.local_path}")
print("Документов выбрано:", len(SOURCES))
for version in SELECTED_VERSIONS[:10]:
    print(f"  {version_title.get(version.id, '?')!r} <- {version.local_path}")
if len(SELECTED_VERSIONS) > 10:
    print(f"  ... и ещё {len(SELECTED_VERSIONS) - 10}")

# --- пре-чек публикации: файлы всех версий снимка должны существовать ---
# Гейт публикации проверяет sha256 КАЖДОЙ версии, включая старые (is_current=False).
# Удалённые с диска файлы исключаем каскадом: версия + её чанки/эмбеддинги + все связи.
unverifiable: list[tuple[str, str]] = []
for version in snapshot.document_versions:
    if not version.local_path:
        continue
    try:
        status = resolver.verify(version.local_path, version.sha256).status
    except Exception:
        status = "missing"
    if status != "verified":
        unverifiable.append((version.id, version.local_path))
if unverifiable:
    print("Исключаю версии без проверяемых файлов (удалённые источники):")
    for version_id, local_path in unverifiable:
        print(f"  {local_path}")

removed_version_ids = {version_id for version_id, _ in unverifiable}
removed_chunk_ids = {
    edge.target_id
    for edge in snapshot.has_chunks
    if edge.source_id in removed_version_ids
}
removed_embedding_ids = {
    edge.target_id
    for edge in snapshot.has_embedding_chunks
    if edge.source_id in removed_chunk_ids
}
still_used = (
    {
        edge.target_id
        for edge in snapshot.has_embedding_chunks
        if edge.source_id not in removed_chunk_ids
    }
    | {edge.target_id for edge in snapshot.has_embedding_assets}
    | {edge.target_id for edge in snapshot.has_summary_embeddings}
)
removed_embedding_ids -= still_used
removed_node_ids = removed_version_ids | removed_chunk_ids | removed_embedding_ids
update: dict[str, Any] = {
    "document_versions": tuple(
        v for v in snapshot.document_versions if v.id not in removed_version_ids
    ),
    "chunks": tuple(c for c in snapshot.chunks if c.id not in removed_chunk_ids),
    "embeddings": tuple(
        e for e in snapshot.embeddings if e.id not in removed_embedding_ids
    ),
}
for field in type(snapshot).model_fields:
    value = getattr(snapshot, field)
    if isinstance(value, tuple) and value and isinstance(value[0], graph.Relation):
        update[field] = tuple(
            edge
            for edge in value
            if edge.source_id not in removed_node_ids
            and edge.target_id not in removed_node_ids
        )
BASE_SNAPSHOT = GraphSnapshot.model_validate(
    snapshot.model_copy(update=update).model_dump(by_alias=True)
)
publication_check = validate_snapshot(
    BASE_SNAPSHOT, mode="publication", media_resolver=resolver
)
if not publication_check.is_valid:
    for issue in publication_check.issues[:20]:
        print(" ", issue.severity, issue.code, issue.table, issue.record_id, issue.message)
    raise RuntimeError("Снимок после исключения не проходит гейт публикации")
print(
    f"База эксперимента: версий {len(BASE_SNAPSHOT.document_versions)} "
    f"(исключено {len(removed_version_ids)}), чанков {len(BASE_SNAPSHOT.chunks)}"
)

# --- Ollama-адаптер: эмбеддинги чанков и семантическая нарезка ---
ollama_base_url = os.environ.get("MVP_V2_OLLAMA_BASE_URL", "").strip()
if not ollama_base_url:
    raise RuntimeError("MVP_V2_OLLAMA_BASE_URL не задан в .env: эмбеддинги невозможны")
allowed_lan_hosts = frozenset(
    host.strip()
    for host in os.environ.get("MVP_V2_OLLAMA_ALLOWED_LAN_HOSTS", "").split(",")
    if host.strip()
)
embedding_model = OllamaAdapter(
    LocalHttpTransport(
        ollama_base_url,
        timeout_seconds=OLLAMA_TIMEOUT_SECONDS,
        allowed_lan_hosts=allowed_lan_hosts,
    ),
    generation_temperature=GENERATION_TEMPERATURE,
)

# --- каталог прогона (создаётся позже, при сборке/сохранении) ---
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S.%f") + "Z"
RUN_DIR = RUNS_ROOT / datetime.now(UTC).strftime("%Y-%m-%d") / RUN_ID
print("Каталог прогона:", RUN_DIR)

SOURCE_DATABASE_NAME = None -> выбрана новейшая БД по built_at
Database   : clean-v9-onboarding.lbdb
Source dump: dump-000058 run 0863af4aacf247965ae5ff2fbcfd3563f1033e8c05c6a59b0b162b5772d20456
Space      : {'provider': 'ollama', 'model': 'bge-m3', 'model_version': '790764642607', 'dimension': 1024}
Snapshot   : документов 30 | версий 33 | чанков 209 | эмбеддингов 223
санитизирован (control-символы): documents/doc-14af693dd9ab78707290fed0.md
санитизирован (control-символы): documents/doc-9dd104d5e1252492b106544a.md
санитизирован (control-символы): documents/doc-d0f463a85c46653cb6feda59.md
санитизирован (control-символы): documents/doc-d202ce979a0c29e56df87d13.md
Документов выбрано: 29
  'Чеклист легенды (для Киры)' <- documents/doc-09ee24d8260e884437e55180.md
  'Гайд по ведению телеграма' <- documents/doc-10dcf72135fc08a6895f086e.md
  'Инструкция по онбордингу' <- documents/doc-14af693dd9ab78707290fed0.md
  'Frontend: day1 program' <- documents/doc-2cfc04955e89dd87b16613ac.md
  'VEU: 

## 5. Стратегии чанкинга (langchain)


Каждая стратегия - фабрика `параметры -> (text -> tuple[str, ...])`. Все получают
один и тот же сырой текст (как продакшен-инжест), поэтому сравнение честное.

Пропущенные стратегии статьи:
- **#5 hierarchical parent-child** - нужен merge детей в родителей на стороне
  retrieval; проектный `SearchService` этого не делает, а код мы не меняем.
- **#6 agentic LLM-driven** - недетерминированно и дорого (статья сама предупреждает);
  в воспроизводимых экспериментах не участвует.
- **#7 table-preserving** - реализована приближенно как `markdown-blocks`
  (таблицы и code-fence - атомарные блоки).
- **#2 sentence-window** - реализована только со стороны инжеста (мелкие чанки);
  окно вокруг найденного чанка уже добавляет продакшен-гидратор evidence.

In [18]:
import re
# --- 1. baseline: продакшен-нарезка проекта (контроль) ---
def make_baseline(params: dict[str, Any]):
    policy = ChunkPolicy(
        max_characters=params["max_characters"],
        overlap_characters=params["overlap_characters"],
    )
    return lambda text: tuple(split_text(text, policy))


# --- 2. fixed-size: чистое окно фиксированного размера с перекрытием ---
def make_fixed(params: dict[str, Any]):
    splitter = CharacterTextSplitter(
        separator="",
        chunk_size=params["chunk_size"],
        chunk_overlap=params["chunk_overlap"],
    )
    return lambda text: tuple(splitter.split_text(text))


# --- 3. sentence-window: окна из N предложений с перекрытием ---
_SENTENCE_END = re.compile(r"(?<=[.!?…])\s+")


def make_sentence_window(params: dict[str, Any]):
    max_sentences = params["max_sentences"]
    step = max(1, max_sentences - params["overlap_sentences"])

    def split(text: str) -> tuple[str, ...]:
        sentences = [
            sentence
            for block in re.split(r"\n\s*\n", text)
            for sentence in _SENTENCE_END.split(block)
            if sentence.strip()
        ]
        return tuple(
            " ".join(sentences[start : start + max_sentences])
            for start in range(0, len(sentences), step)
        )

    return split


# --- 4. markdown-recursive: рекурсивный markdown-aware сплиттер ---
def make_markdown_recursive(params: dict[str, Any]):
    splitter = RecursiveCharacterTextSplitter.from_language(
        Language.MARKDOWN,
        chunk_size=params["chunk_size"],
        chunk_overlap=params["chunk_overlap"],
    )
    return lambda text: tuple(splitter.split_text(text))


# --- 5. markdown-headers: секции по H1/H2/H3 + хлебные крошки заголовков ---
def make_markdown_headers(params: dict[str, Any]):
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
        strip_headers=True,
    )
    fallback = RecursiveCharacterTextSplitter.from_language(
        Language.MARKDOWN,
        chunk_size=params["chunk_size"],
        chunk_overlap=params["chunk_overlap"],
    )

    def split(text: str) -> tuple[str, ...]:
        chunks: list[str] = []
        for section in header_splitter.split_text(text):
            breadcrumb = " > ".join(
                section.metadata[key]
                for key in ("h1", "h2", "h3")
                if section.metadata.get(key)
            )
            body = section.page_content.strip()
            if not body:
                continue
            pieces = (
                fallback.split_text(body)
                if len(body) > params["chunk_size"]
                else [body]
            )
            for piece in pieces:
                chunks.append(f"{breadcrumb}\n\n{piece}" if breadcrumb else piece)
        return tuple(chunks)

    return split


# --- 6. markdown-blocks: таблицы и code-fence - атомарные блоки ---
def _markdown_blocks(text: str) -> list[str]:
    """Бьёт по пустым строкам, но не внутри fenced-блоков; таблица - один блок."""
    blocks: list[str] = []
    current: list[str] = []
    in_fence = False
    for line in text.splitlines():
        if line.strip().startswith("```"):
            in_fence = not in_fence
            current.append(line)
            continue
        if not line.strip() and not in_fence:
            if current:
                blocks.append("\n".join(current))
                current = []
            continue
        current.append(line)
    if current:
        blocks.append("\n".join(current))
    return [block for block in blocks if block.strip()]


def make_markdown_blocks(params: dict[str, Any]):
    max_characters = params["max_characters"]
    fallback = RecursiveCharacterTextSplitter.from_language(
        Language.MARKDOWN, chunk_size=max_characters, chunk_overlap=0
    )

    def split(text: str) -> tuple[str, ...]:
        packed: list[str] = []
        current = ""
        for block in _markdown_blocks(text):
            candidate = f"{current}\n\n{block}" if current else block
            if len(candidate) <= max_characters:
                current = candidate
                continue
            if current:
                packed.append(current)
            if len(block) > max_characters:  # слишком большой блок - всё же режем
                packed.extend(fallback.split_text(block))
                current = ""
            else:
                current = block
        if current:
            packed.append(current)
        return tuple(packed)

    return split


# --- 7. semantic: границы по косинусной дистанции соседних предложений ---
class OllamaEmbeddingsAdapter(Embeddings):
    """Адаптер проектного OllamaAdapter под интерфейс langchain Embeddings."""

    def __init__(self, model: Any, model_name: str) -> None:
        self._model = model
        self._model_name = model_name

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = self._model.embed(texts=tuple(texts), model=self._model_name)
        return [list(vector) for vector in vectors]

    def embed_query(self, text: str) -> list[float]:
        return list(self._model.embed(texts=(text,), model=self._model_name)[0])


def make_semantic(params: dict[str, Any]):
    chunker = SemanticChunker(
        OllamaEmbeddingsAdapter(
            embedding_model, database_manifest.embedding_space.model
        ),
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=params["breakpoint_threshold_amount"],
    )
    return lambda text: tuple(chunker.split_text(text))


STRATEGY_FACTORIES = {
    "baseline-1200-120": ("Контроль: продакшен split_text проекта", make_baseline),
    "fixed-512-64": ("Статья #1: окно фиксированного размера + overlap", make_fixed),
    "sentence-window": ("Статья #2: окна предложений (small-to-big)", make_sentence_window),
    "markdown-recursive": ("Статья #3: рекурсивный markdown-aware", make_markdown_recursive),
    "markdown-headers": ("Статья #3: по заголовкам + хлебные крошки", make_markdown_headers),
    "markdown-blocks": ("Статья #7: блоки; таблицы/код целыми", make_markdown_blocks),
    "semantic-ollama": ("Статья #4: семантические границы (Ollama)", make_semantic),
}

for name in STRATEGIES:
    description, _ = STRATEGY_FACTORIES[name]
    print(f"  {name:20s} {description}")
print("Стратегии готовы")

  baseline-1200-120    Контроль: продакшен split_text проекта
  fixed-512-64         Статья #1: окно фиксированного размера + overlap
  sentence-window      Статья #2: окна предложений (small-to-big)
  markdown-recursive   Статья #3: рекурсивный markdown-aware
  markdown-headers     Статья #3: по заголовкам + хлебные крошки
  markdown-blocks      Статья #7: блоки; таблицы/код целыми
  semantic-ollama      Статья #4: семантические границы (Ollama)
Стратегии готовы


## 6. Превью: статистика чанков по стратегиям

Быстрая проверка ДО дорогой сборки: сколько чанков даёт каждая стратегия,
размеры (min/среднее/max) и первые чанки первого документа. `semantic-ollama`
считается только по первому документу - она эмбеддит каждое предложение.

In [19]:
PREVIEW_ROWS: list[dict[str, Any]] = []
for name in STRATEGIES:
    _, factory = STRATEGY_FACTORIES[name]
    splitter = factory(STRATEGY_PARAMS[name])
    versions = SELECTED_VERSIONS[:1] if name == "semantic-ollama" else SELECTED_VERSIONS
    counts: list[int] = []
    lengths: list[int] = []
    started_at = time.perf_counter()
    for version in versions:
        pieces = [piece for piece in splitter(SOURCES[version.id]) if piece.strip()]
        counts.append(len(pieces))
        lengths.extend(len(piece) for piece in pieces)
    elapsed_s = round(time.perf_counter() - started_at, 2)
    PREVIEW_ROWS.append(
        {
            "strategy": name,
            "documents": len(versions),
            "chunks": sum(counts),
            "min_chars": min(lengths) if lengths else 0,
            "mean_chars": round(sum(lengths) / len(lengths)) if lengths else 0,
            "max_chars": max(lengths) if lengths else 0,
            "seconds": elapsed_s,
        }
    )
    row = PREVIEW_ROWS[-1]
    note = " (первый документ)" if name == "semantic-ollama" else ""
    print(
        f"{name:20s} чанков {row['chunks']:4d} | min {row['min_chars']:4d} "
        f"| mean {row['mean_chars']:4d} | max {row['max_chars']:4d} "
        f"| {elapsed_s:6.2f} c{note}"
    )

first_version = SELECTED_VERSIONS[1]
print(f"\nПервые чанки документа {version_title.get(first_version.id, '?')!r}:")
for name in STRATEGIES:
    _, factory = STRATEGY_FACTORIES[name]
    splitter = factory(STRATEGY_PARAMS[name])
    pieces = [piece for piece in splitter(SOURCES[first_version.id]) if piece.strip()]
    print(f"\n--- {name} ({len(pieces)} чанков) ---")
    for position, piece in enumerate(pieces[:2]):
        preview = piece[:PREVIEW_CHARACTERS].replace("\n", "\\n")
        print(f"  [{position}] len={len(piece):4d}  {preview!r}")
print("\nЭто только превью: ничего не записано")

baseline-1200-120    чанков  209 | min  159 | mean 1111 | max 1199 |   0.00 c
fixed-512-64         чанков  480 | min  120 | mean  499 | max  512 |   0.09 c
sentence-window      чанков 1303 | min   11 | mean  236 | max 1734 |   0.00 c
markdown-recursive   чанков  236 | min    3 | mean  938 | max 1199 |   0.00 c
markdown-headers     чанков  259 | min   71 | mean  887 | max 1307 |   0.01 c
markdown-blocks      чанков  206 | min   50 | mean 1020 | max 1200 |   0.00 c
semantic-ollama      чанков    1 | min  348 | mean  348 | max  348 |   0.00 c (первый документ)

Первые чанки документа 'Гайд по ведению телеграма':

--- baseline-1200-120 (2 чанков) ---
  [0] len=1195  '# Гайд по ведению телеграма\\n\\n1. Чтобы было проще ориентироваться в огромном количестве рекрутеров, добавляйте их в контакты и переименовывайте по шаблону:\\n\\n**[НАЗВАНИЕ КОМПАНИИ] Имя рекрутера**\\n\\n2. Чт'
  [1] len= 253  'сите сообщать о новых позициях в компании и после того, как диалог логически заканчивается, убира

## 7. Сборка экспериментальных БД

Для каждой стратегии:
1. нарезаем выбранные документы, строим чанки продакшен-функцией `make_chunks`;
2. собираем вариантный снимок: заменены только чанки выбранных версий и их
   эмбеддинги; документы, версии, ассеты и их эмбеддинги не трогаем;
3. доэмбеддиваем новые чанки через `build_embeddings` (mode="missing");
4. публикуем вариантный дамп в `RUN_DIR/dumps` (через продакшен-гейт валидации);
5. строим `RUN_DIR/db/<стратегия>.lbdb` через продакшен `build_database`.

Ошибка одной стратегии не роняет остальные: она печатается и попадает в
`BUILD_FAILURES`. Повторный запуск этой ячейки в том же прогоне упадёт с
"target already exists" - сделайте Run All (новый RUN_ID).

In [20]:
def build_variant_snapshot(texts_by_version: dict[str, tuple[str, ...]]) -> GraphSnapshot:
    """Заменяет чанки выбранных версий; остальной снимок не трогает."""
    replaced_chunk_ids: set[str] = set()
    new_chunks: list[Any] = []
    new_links: list[Any] = []
    for version in SELECTED_VERSIONS:
        pieces = texts_by_version.get(version.id)
        if pieces is None:
            continue
        for edge in BASE_SNAPSHOT.has_chunks:
            if edge.source_id == version.id:
                replaced_chunk_ids.add(edge.target_id)
        chunks = make_chunks(version.id, pieces)
        new_chunks.extend(chunks)
        new_links.extend(
            graph.HasChunk.model_validate({"from": version.id, "to": chunk.id})
            for chunk in chunks
        )
    orphan_embedding_ids = {
        edge.target_id
        for edge in BASE_SNAPSHOT.has_embedding_chunks
        if edge.source_id in replaced_chunk_ids
    }
    still_used = (
        {
            edge.target_id
            for edge in BASE_SNAPSHOT.has_embedding_chunks
            if edge.source_id not in replaced_chunk_ids
        }
        | {edge.target_id for edge in BASE_SNAPSHOT.has_embedding_assets}
        | {edge.target_id for edge in BASE_SNAPSHOT.has_summary_embeddings}
    )
    dropped_embedding_ids = orphan_embedding_ids - still_used
    variant = BASE_SNAPSHOT.model_copy(
        update={
            "chunks": tuple(
                chunk for chunk in BASE_SNAPSHOT.chunks if chunk.id not in replaced_chunk_ids
            )
            + tuple(new_chunks),
            "has_chunks": tuple(
                edge
                for edge in BASE_SNAPSHOT.has_chunks
                if edge.target_id not in replaced_chunk_ids
            )
            + tuple(new_links),
            "embeddings": tuple(
                item for item in BASE_SNAPSHOT.embeddings if item.id not in dropped_embedding_ids
            ),
            "has_embedding_chunks": tuple(
                edge
                for edge in BASE_SNAPSHOT.has_embedding_chunks
                if edge.source_id not in replaced_chunk_ids
            ),
        }
    )
    # Каноническая форма: те же валидаторы и сортировка, что при загрузке дампа.
    return GraphSnapshot.model_validate(variant.model_dump(by_alias=True))


BUILT_DATABASES: list[dict[str, Any]] = []
BUILD_FAILURES: list[dict[str, str]] = []

if not BUILD_DATABASES:
    print("BUILD_DATABASES = False: сборка пропущена (только превью чанков)")
else:
    configure_ladybug_home(config)
    (RUN_DIR / "dumps").mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "db").mkdir(parents=True, exist_ok=True)
    space = database_manifest.embedding_space
    identity = ModelIdentity(
        provider=space.provider,
        model=space.model,
        model_version=space.model_version,
        dimension=space.dimension,
    )
    publication = SnapshotPublicationService(FilesystemDumpRepository())

    for name in STRATEGIES:
        started_at = time.perf_counter()
        try:
            splitter = STRATEGY_FACTORIES[name][1](STRATEGY_PARAMS[name])
            texts_by_version: dict[str, tuple[str, ...]] = {}
            for version in SELECTED_VERSIONS:
                pieces = tuple(
                    piece for piece in splitter(SOURCES[version.id]) if piece.strip()
                )
                if not pieces:
                    raise ValueError(
                        f"0 чанков для {version.local_path}: проверьте параметры"
                    )
                texts_by_version[version.id] = pieces

            variant = build_variant_snapshot(texts_by_version)
            request = EmbeddingRequest(
                identity=identity,
                authorization=ModelAuthorization(allow=True, operation="embedding"),
                mode="missing",
                generation_token=f"chunking-experiment-{name}",
                created_at=datetime.now(UTC),
            )
            embedded = build_embeddings(variant, request, embedding_model)
            variant_dump_manifest = publication.publish(
                snapshot=embedded,
                dumps_root=RUN_DIR / "dumps",
                published_at=datetime.now(UTC),
                media_resolver=resolver,
            )
            variant_dump_path = (
                RUN_DIR / "dumps" / f"dump-{variant_dump_manifest.dump_number:06d}"
            )
            target = RUN_DIR / "db" / f"{name}.lbdb"
            manifest = build_database(
                dump_path=variant_dump_path,
                target=target,
                database_name=name,
                space=space,
                vision_provider=database_manifest.vision_provider,
                vision_model=database_manifest.vision_model,
                vision_model_version=database_manifest.vision_model_version,
            )
            BUILT_DATABASES.append(
                {
                    "strategy": name,
                    "database_path": target,
                    "manifest": manifest,
                    "snapshot": embedded,
                    "chunk_count": sum(len(p) for p in texts_by_version.values()),
                    "elapsed_s": round(time.perf_counter() - started_at, 1),
                }
            )
            print(
                f"[ok] {name}: чанков {BUILT_DATABASES[-1]['chunk_count']}, "
                f"{BUILT_DATABASES[-1]['elapsed_s']} c -> {target}"
            )
        except Exception as failure:  # noqa: BLE001 - показываем и идём дальше
            BUILD_FAILURES.append(
                {
                    "strategy": name,
                    "type": type(failure).__name__,
                    "message": str(failure),
                }
            )
            print(f"[FAIL] {name}: {type(failure).__name__}: {failure}")

    print(
        f"\nСобрано {len(BUILT_DATABASES)} из {len(STRATEGIES)} стратегий; "
        f"ошибок: {len(BUILD_FAILURES)}"
    )

[FAIL] baseline-1200-120: SnapshotPublicationError: snapshot or referenced files failed publication validation
[FAIL] fixed-512-64: SnapshotPublicationError: snapshot or referenced files failed publication validation
[FAIL] sentence-window: SnapshotPublicationError: snapshot or referenced files failed publication validation
[FAIL] markdown-recursive: SnapshotPublicationError: snapshot or referenced files failed publication validation
[FAIL] markdown-headers: SnapshotPublicationError: snapshot or referenced files failed publication validation
[FAIL] markdown-blocks: SnapshotPublicationError: snapshot or referenced files failed publication validation
[FAIL] semantic-ollama: OllamaError: local Ollama response exceeds configured limit

Собрано 0 из 7 стратегий; ошибок: 7


## 8. Сравнение поиска по всем версиям БД

Одни и те же вопросы -> каждая собранная БД -> тот же `SearchService`,
тот же порог и лимит. Сначала сводная таблица top-1, потом детали.

In [11]:
SEARCH_RESULTS: list[dict[str, Any]] = []

if not RUN_SEARCH:
    print("RUN_SEARCH = False: сравнение поиска пропущено")
elif not BUILT_DATABASES:
    print("Нет собранных БД: сравнение поиска пропущено")
else:
    configure_ladybug_home(config)
    ladybug_module = importlib.import_module("ladybug")
    for built in BUILT_DATABASES:
        name = built["strategy"]
        manifest = built["manifest"]
        variant_snapshot = built["snapshot"]
        database = ladybug_module.Database(
            str(built["database_path"] / "database.lbug"), read_only=True
        )
        try:
            warmup_connection = ladybug_module.Connection(database)
            try:
                warmup_result = warmup_connection.execute("LOAD vector")
                warmup_result.close()
            finally:
                warmup_connection.close()

            for index, question in enumerate(MY_QUESTIONS, start=1):
                connection = ladybug_module.Connection(database)
                evidence: tuple = ()
                failure_info: str | None = None
                started_at = time.perf_counter()
                try:
                    service = SearchService(
                        embedding_model,
                        LadybugAnn(connection, manifest),
                        manifest,
                        variant_snapshot,
                    )
                    request = SearchRequest(
                        question=question,
                        space=manifest.embedding_space,
                        purposes=(
                            EmbeddingPurpose.CHUNK_RETRIEVAL,
                            EmbeddingPurpose.ASSET_DESCRIPTION_RETRIEVAL,
                        ),
                        limit=EVIDENCE_LIMIT,
                        minimum_score=MINIMUM_SIMILARITY_SCORE,
                        authorization=ModelAuthorization(
                            allow=True, operation="embedding"
                        ),
                    )
                    evidence = service.search(request)
                except Exception as failure:  # noqa: BLE001
                    failure_info = f"{type(failure).__name__}: {failure}"
                finally:
                    connection.close()
                latency_ms = round((time.perf_counter() - started_at) * 1_000, 1)
                SEARCH_RESULTS.append(
                    {
                        "question": question,
                        "strategy": name,
                        "evidence": [
                            item.model_dump(mode="json") for item in evidence
                        ],
                        "latency_ms": latency_ms,
                        "error": failure_info,
                    }
                )
        finally:
            database.close()
        print(f"[ok] поиск по {name}: {len(MY_QUESTIONS)} вопросов")

print("\n=== Сводка top-1 по вопросам ===")
for index, question in enumerate(MY_QUESTIONS, start=1):
    print(f"\nQ{index}: {question}")
    rows = [row for row in SEARCH_RESULTS if row["question"] == question]
    for row in rows:
        label = f"  {row['strategy']:20s}"
        if row["error"]:
            print(f"{label} ERROR {row['error']}")
            continue
        if not row["evidence"]:
            print(f"{label} (пусто после порога {MINIMUM_SIMILARITY_SCORE})")
            continue
        top = row["evidence"][0]
        position = (
            top["matched_chunk_position"] + 1
            if top["matched_chunk_position"] is not None
            else "-"
        )
        print(
            f"{label} score={top['score']:.4f} {top['document_title']!r} "
            f"chunk {position}/{top['document_chunk_count']}"
        )

print("\n=== Детали: все evidence по стратегиям ===")
for row in SEARCH_RESULTS:
    print(f"\n--- {row['strategy']} | {row['question'][:60]} ---")
    if row["error"]:
        print(f"    ERROR {row['error']}")
        continue
    for item in row["evidence"]:
        position = (
            item["matched_chunk_position"] + 1
            if item["matched_chunk_position"] is not None
            else "-"
        )
        print(
            f"    score={item['score']:.4f} {item['document_title']!r} "
            f"chunk {position}/{item['document_chunk_count']}"
        )
        preview = item["text"][:PREVIEW_CHARACTERS].replace("\n", "\\n")
        print(f"    text: {preview!r}")

Нет собранных БД: сравнение поиска пропущено

=== Сводка top-1 по вопросам ===

Q1: Что нужно для того, чтобы мы начали работу над твоим резюме?

Q2: Где купить прокси?

Q3: Сколько рабочих почт, номеров, SIM-карт, HH и Telegram нужно создать?

=== Детали: все evidence по стратегиям ===


## 9. Сохранение прогона

`run.json` (настройки + источник), `results.json` (поиск по стратегиям),
`chunk_stats.json` (превью + итоги сборки). Пути к собранным БД - в `run.json`.

In [12]:
RUN_DIR.mkdir(parents=True, exist_ok=True)


def write_json_atomic(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


run_metadata = {
    "run_id": RUN_ID,
    "notebook": "chunking_strategies_experiments.ipynb",
    "saved_at": datetime.now(UTC).isoformat(),
    "config": {
        "source_database_name": database_path.name,
        "source_dump_run_id": dump_manifest.run_id,
        "embedding_space": database_manifest.embedding_space.model_dump(mode="json"),
        "strategies": {name: STRATEGY_PARAMS[name] for name in STRATEGIES},
        "max_documents": MAX_DOCUMENTS,
        "documents": [
            version_title.get(version.id, version.id) for version in SELECTED_VERSIONS
        ],
        "evidence_limit": EVIDENCE_LIMIT,
        "minimum_similarity_score": MINIMUM_SIMILARITY_SCORE,
        "questions": list(MY_QUESTIONS),
    },
    "built_databases": [
        {
            "strategy": built["strategy"],
            "database_path": str(built["database_path"]),
            "chunk_count": built["chunk_count"],
            "elapsed_s": built["elapsed_s"],
        }
        for built in BUILT_DATABASES
    ],
    "build_failures": BUILD_FAILURES,
}
write_json_atomic(RUN_DIR / "run.json", run_metadata)
write_json_atomic(RUN_DIR / "results.json", SEARCH_RESULTS)
write_json_atomic(RUN_DIR / "chunk_stats.json", PREVIEW_ROWS)
print("Прогон сохранён:", RUN_DIR)
print("Сравнивайте results.json между прогонами: одни вопросы + один код -> один топ-N")

Прогон сохранён: /Users/zero/GITHUB/vue-graph-rag/mvp_v2/runtime/debug/chunking-experiments/2026-10-20/20261020T193743.289705Z
Сравнивайте results.json между прогонами: одни вопросы + один код -> один топ-N


## Мои заметки

- Какая стратегия лучше по вопросам поддержки:
- Что попробовать дальше (параметры, пороги, вопросы):